# Week 3, day 1 (afternoon) — Extra practice 04 SOLUTIONS: loc and iloc   (L01/L02)

Executed in the lab image against the real `../data/sales.csv`. Every quoted
number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 04 — loc and iloc. Run this once.
import pandas as pd

sales = pd.read_csv("../data/sales.csv")

# Sorted by Sales descending -- the labels are now out of order.
ranked = sales.sort_values("Sales", ascending=False)

print("sales index starts:", list(sales.index[:5]))
print("ranked index starts:", list(ranked.index[:5]))

### Question 1

Top three sales are OrderIDs `10852`, `293`, `15622`. -> labels `[155, 229, 237]`, not `[0, 1, 2]`.

Sorting reordered the rows and carried each one's original label along.
The biggest sale was row 155 of the file and it still is — it has just been
moved to the front.

So the frame now has one order that is *first* and a different order that
is *labelled 0*. Q2 is the consequence.

In [ ]:
print(ranked.head(3)[["OrderID", "Sales"]])
print()
print("labels:", list(ranked.index[:3]))

# Sorting moved the rows and carried their original labels with them.

### Question 2

`ranked.iloc[0]` -> OrderID `10852`, the `12450.48` sale. `ranked.loc[0]` -> OrderID `8710`, worth `151.35`.

Same frame, same integer `0`, two different orders — and an eighty-fold
difference in value between them.

`iloc[0]` means 'the first row as currently arranged', which after sorting
is the biggest sale. `loc[0]` means 'the row labelled 0', which is still the
first row of the original file and is now sitting somewhere in the middle.

Neither call raises. If you wrote `ranked.loc[0]` intending 'the top
seller', you get a real order with real numbers and no indication that it is
the wrong one.

In [ ]:
print("ranked.iloc[0] OrderID:", ranked.iloc[0]["OrderID"], "-> the biggest sale")
print("ranked.loc[0]  OrderID:", ranked.loc[0, "OrderID"], "-> the original first row")
print()
print("biggest sale amount:", ranked.iloc[0]["Sales"])
print("original row 0 sale:", ranked.loc[0, "Sales"])

### Question 3

`head(5)` and `iloc[:5]` -> identical, `.equals()` is `True`.

`head(n)` is `iloc[:n]` with a friendlier name — both positional, both
relative to the current arrangement. Neither has anything to do with the
labels.

In [ ]:
a = ranked.head(5)
b = ranked.iloc[:5]
print(a[["OrderID", "Sales"]])
print()
print("head(5) equals iloc[:5]:", a.equals(b))

### Question 4

`iloc[:3][["Region", "Sales"]]` and `iloc[:3, [2, 7]]` -> identical.

They agree because `Region` and `Sales` genuinely are at positions 2 and 7
— which you can only know by printing the column list first.

Prefer the names. Column positions are stable only until someone adds a
column upstream, and when that happens `iloc[:3, [2, 7]]` keeps working and
silently returns two different columns. Nothing about the code changes and
nothing raises.

In [ ]:
print("columns:", list(sales.columns))
print()
by_name = ranked.iloc[:3][["Region", "Sales"]]
by_pos = ranked.iloc[:3, [2, 7]]
print(by_name)
print()
print("same:", by_name.equals(by_pos))

### Question 5

After `reset_index(drop=True)` labels are `[0, 1, 2]` -> `loc[0]` and `iloc[0]` both give OrderID `10852`.

The labels have been renumbered to match the new order, so the coincidence
from worksheet 04 Q2 is restored and the two forms agree again.

`drop=True` discards the old labels entirely. That is right when the sort
order is the meaning — a ranked leaderboard — and wrong when you still need
to trace a row back to its position in the source file, because the
information is gone for good.

In [ ]:
renumbered = ranked.reset_index(drop=True)
print("labels:", list(renumbered.index[:3]))
print()
print("loc[0]: ", renumbered.loc[0, "OrderID"])
print("iloc[0]:", renumbered.iloc[0]["OrderID"])
print("agree:", renumbered.loc[0, "OrderID"] == renumbered.iloc[0]["OrderID"])

### Question 6

`sales.loc[mask, ["OrderID", "Sales"]]` -> `5` orders over 8000. -> labels `155, 190, 228, 229, 237`.

Rows and columns selected in one call. Note the results come back in
**label order**, not sorted by `Sales` — filtering preserves the original
arrangement and does no sorting of its own.

Those are the same five labels as the top of `ranked`, in a different
order, which is a useful cross-check.

In [ ]:
print(sales.loc[sales["Sales"] > 8000, ["OrderID", "Sales"]])
print()
print("rows:", (sales["Sales"] > 8000).sum())

### Question 7

`sales.loc[0:4]` -> **5 rows** (`0,1,2,3,4`). `sales.iloc[0:4]` -> **4 rows** (`0,1,2,3`).

Identical-looking expressions on a frame whose labels are `0..299`, and
they differ by one row.

`.loc` includes the end **label**; `.iloc` excludes the end **position**.
On a default index the two spellings look interchangeable, which is exactly
why this costs people a row so often — the code reads correctly either way.

A report built with `iloc[0:n]` when `loc[0:n]` was meant is short by one
line, every time, silently.

In [ ]:
print("loc[0:4] rows: ", len(sales.loc[0:4]), "labels:", list(sales.loc[0:4].index))
print("iloc[0:4] rows:", len(sales.iloc[0:4]), "labels:", list(sales.iloc[0:4].index))

# .loc includes the end LABEL (4), .iloc excludes the end POSITION (4).

### Question 8

`ranked.loc[500, "OrderID"]` -> **raises** `KeyError: 500`. -> the highest label in the file is `299`.

There is no row labelled 500 — the file has 300 rows labelled 0 to 299.

This is the well-behaved failure. Compare it with Q2, where an in-range
label returned the wrong row perfectly happily. `.loc` protects you from
labels that do not exist and cannot protect you from labels that exist and
mean something other than what you assumed.

In [ ]:
print("highest label in the file:", sales.index.max())
print(ranked.loc[500, "OrderID"])